# Keyence tile illumination correction — visual comparison

Reconstructs the 3-tile mosaic for one well (`20260702_hotchem_24hpf_plate01_A02`) from the RAW tiles, then compares:
- **Raw** (no correction)
- **(1) Seam-sampling** — per-tile multiplicative gain from the overlap-region medians (current implementation)
- **(2) Tile-wide sampling** — per-tile multiplicative gain from the WHOLE-tile medians (not seam-restricted)
- **(3) Affine (offset + linear ramp)** — per-tile `A + B*x` fit

Each panel shows the composite image plus a mean-intensity profile along the stitching (long) axis, with seam positions marked.

In [ ]:
import sys
sys.path.insert(0, '/net/trapnell/vol1/home/nlammers/projects/repositories/morphseq/src')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import skimage.io as skio
from data_pipeline.shared.path_roots import resolve_under_input_root

ROOT = '/net/trapnell/vol1/home/nlammers/projects/data/morphseq'
INV = f'{ROOT}/pipeline_output/acquisition/20260702_hotchem_24hpf_plate01/ingest_metadata/acquisition_inventory__keyence.csv'
WELL_SUFFIX = '_A02'
Z_INDEX = 6  # a mid-stack slice (more in-focus signal than z=1)

In [ ]:
# --- Load the raw tiles for this well/z, ordered LEFT->RIGHT by stage position ---
df = pd.read_csv(INV)
well = [w for w in df.well_id.unique() if w.endswith(WELL_SUFFIX)][0]
sub = df[(df.well_id == well) & (df.time_index == df.time_index.min())]
umpp = float(sub.micrometers_per_pixel.iloc[0])
tile_w = int(sub.image_width_px.iloc[0])

# stage x-offset per tile_id -> physical left->right order
per = sub.groupby('tile_id')[['stage_x_nm']].first()
per['x_px'] = per['stage_x_nm'] / 1000.0 / umpp
per['x_px'] -= per['x_px'].min()
order = per.sort_values('x_px').index.tolist()[::-1]   # tile_ids LEFT->RIGHT (stage-x reversed to match display)
offsets = per.loc[order, 'x_px'].to_numpy(); offsets = offsets.max() - offsets  # re-anchor left=0 after reversal
print('tile_ids left->right:', order, ' offsets(px):', offsets.round(0))

zsub = sub[sub.z_index == Z_INDEX]
tiles = []  # left->right, float
for tid in order:
    r = zsub[zsub.tile_id == tid].iloc[0]
    p = resolve_under_input_root(r.source_tiff_path, input_root=ROOT, scope_label='t', full_root_fallback=True)
    tiles.append(skio.imread(str(p)).astype(np.float64))
print('tile shapes:', [t.shape for t in tiles], '  tile_width:', tile_w, '  overlap:', tile_w - int(round(offsets[1]-offsets[0])))

In [ ]:
# --- Compositing helper: place gain/affine-corrected tiles at integer offsets, averaging overlaps ---
step = int(round(offsets[1] - offsets[0]))            # px between adjacent tile origins
overlap = tile_w - step
H = tiles[0].shape[0]
W = step * (len(tiles) - 1) + tile_w

def composite(corrected_tiles):
    acc = np.zeros((H, W)); cnt = np.zeros((H, W))
    for i, t in enumerate(corrected_tiles):
        x0 = i * step
        acc[:, x0:x0+tile_w] += t
        cnt[:, x0:x0+tile_w] += 1
    cnt[cnt == 0] = 1
    return acc / cnt

def seam_positions():
    # centers of each overlap band in composite coords
    return [i*step + tile_w - overlap/2 for i in range(len(tiles)-1)]

def show(ax_img, ax_prof, img, title):
    ax_img.imshow(img, cmap='gray'); ax_img.set_title(title); ax_img.axis('off')
    prof = np.mean(img, axis=0)                        # mean along SHORT axis -> profile along LONG (stitch) axis
    ax_prof.plot(prof, lw=0.9)
    for s in seam_positions():
        ax_img.axvline(s, color='r', ls='--', lw=0.6)
        ax_prof.axvline(s, color='r', ls='--', lw=0.6)
    ax_prof.set_xlabel('position along stitch axis (px)'); ax_prof.set_ylabel('mean intensity')
    ax_prof.set_title(f'{title} — intensity profile')

In [ ]:
# --- Overlap column slices between adjacent tiles (in each tile's own frame) ---
def overlap_cols():
    # left tile's rightmost `overlap` cols  <->  right tile's leftmost `overlap` cols
    return slice(tile_w - overlap, tile_w), slice(0, overlap)
left_cols, right_cols = overlap_cols()
center = len(tiles) // 2

# ============ (1) SEAM sampling: gain = ratio of OVERLAP medians, chained to center ============
def gains_seam():
    g = [1.0]*len(tiles); g[center] = 1.0
    # walk outward from center
    for i in range(center-1, -1, -1):   # left side
        this_med = np.median(tiles[i][:, left_cols])
        inner_med = np.median(tiles[i+1][:, right_cols])
        g[i] = (inner_med/this_med) * g[i+1]
    for i in range(center+1, len(tiles)):  # right side
        this_med = np.median(tiles[i][:, right_cols])
        inner_med = np.median(tiles[i-1][:, left_cols])
        g[i] = (inner_med/this_med) * g[i-1]
    return g

# ============ (2) TILE-WIDE sampling: gain = center_full_median / tile_full_median ============
def gains_tilewide():
    fm = [np.median(t) for t in tiles]
    return [fm[center]/m for m in fm]

g1 = gains_seam(); g2 = gains_tilewide()
print('(1) seam gains     :', [round(x,3) for x in g1])
print('(2) tile-wide gains:', [round(x,3) for x in g2])

In [ ]:
# ============ (3) AFFINE per tile: correct each tile to center via A + B*x fit ============
# Fit so that, in each overlap, corrected(this) matches the (already-corrected) inner neighbour.
# Model applied to a tile: out = A + B * in. Solve A,B by least squares on the paired overlap pixels
# (this tile's overlap pixels as x, the inner neighbour's corrected overlap pixels as y).
def affine_correct():
    corr = [t.copy() for t in tiles]
    params = [(0.0,1.0)]*len(tiles)
    def fit(x, y):
        x = x.ravel().astype(float); y = y.ravel().astype(float)
        B, A = np.polyfit(x, y, 1)   # y = B*x + A
        return A, B
    for i in range(center-1, -1, -1):
        x = tiles[i][:, left_cols]; y = corr[i+1][:, right_cols]
        A,B = fit(x,y); params[i]=(A,B); corr[i] = A + B*tiles[i]
    for i in range(center+1, len(tiles)):
        x = tiles[i][:, right_cols]; y = corr[i-1][:, left_cols]
        A,B = fit(x,y); params[i]=(A,B); corr[i] = A + B*tiles[i]
    return corr, params

corr_affine, affine_params = affine_correct()
print('(3) affine (A offset, B ramp) per tile:')
for tid, (A,B) in zip(order, affine_params): print(f'   tile {tid}: A={A:.1f}  B={B:.3f}')

In [ ]:
# --- Build all four composites ---
raw_comp     = composite(tiles)
seam_comp    = composite([g1[i]*tiles[i] for i in range(len(tiles))])
tilewide_comp= composite([g2[i]*tiles[i] for i in range(len(tiles))])
affine_comp  = composite(corr_affine)

panels = [('Raw (no correction)', raw_comp),
          ('(1) Seam sampling', seam_comp),
          ('(2) Tile-wide sampling', tilewide_comp),
          ('(3) Affine offset+ramp', affine_comp)]

fig, axes = plt.subplots(2, 4, figsize=(22, 8), gridspec_kw={'height_ratios':[3,1.4]})
for j, (title, img) in enumerate(panels):
    show(axes[0, j], axes[1, j], img, title)
# shared y-limits on profiles for fair comparison
ymax = max(np.mean(p[1], axis=0).max() for p in panels)
for j in range(4): axes[1, j].set_ylim(0, ymax*1.05)
plt.tight_layout(); plt.show()

In [ ]:
# --- Quantify seam discontinuity: intensity STEP across each seam, per method ---
def seam_steps(img):
    prof = np.mean(img, axis=0); steps=[]
    for s in seam_positions():
        s=int(round(s))
        left=np.median(prof[s-20:s-2]); right=np.median(prof[s+2:s+20])
        steps.append(right-left)
    return [round(x,1) for x in steps]
for title,img in panels:
    print(f'{title:28s} seam steps: {seam_steps(img)}   (closer to 0 = smoother)')

In [ ]:
# ==========================================================================================
# CELL 2 — fancier per-tile illumination corrections
#   (i)  Linear-along-stitch-axis: model each tile's illumination as a 1-D linear ramp across the
#        stitch axis (fit from the tile's own column-medians), divide it out, rescale to a common
#        level. Removes WITHIN-tile gradient, not just seam offset.
#   (ii) Local Gaussian background division: estimate each tile's smooth illumination field as a
#        heavily-Gaussian-blurred copy of itself (large sigma => background only), divide it out.
#        This is a flat-field / rolling-illumination estimate — fully local, no seam assumptions.
# ==========================================================================================
from scipy.ndimage import gaussian_filter

# ---- (i) linear-along-stitch-axis correction ----
def correct_linear_axis(tiles):
    out = []
    global_target = np.median(np.concatenate([t.ravel() for t in tiles]))
    for t in tiles:
        colmed = np.median(t, axis=0)                  # per-column median along stitch axis
        x = np.arange(len(colmed))
        b, a = np.polyfit(x, colmed, 1)                # illumination ramp a + b*x
        ramp = (a + b*x)[None, :]
        ramp = np.clip(ramp, 1e-6, None)
        out.append(t / ramp * global_target)           # divide out ramp, restore common level
    return out

# ---- (ii) local Gaussian background division ----
def correct_gaussian(tiles, sigma_frac=0.5):
    # sigma large relative to tile so the blur captures illumination, not embryo detail
    out = []
    global_target = np.median(np.concatenate([t.ravel() for t in tiles]))
    for t in tiles:
        sigma = sigma_frac * min(t.shape)
        field = gaussian_filter(t, sigma=sigma)
        field = np.clip(field, 1e-6, None)
        out.append(t / field * global_target)
    return out

corr_linear = correct_linear_axis(tiles)
corr_gauss  = correct_gaussian(tiles, sigma_frac=0.5)

panels2 = [('Raw', composite(tiles)),
           ('(i) Linear-axis ramp', composite(corr_linear)),
           ('(ii) Gaussian flat-field', composite(corr_gauss))]

fig, axes = plt.subplots(2, 3, figsize=(18, 8), gridspec_kw={'height_ratios':[3,1.4]})
for j,(title,img) in enumerate(panels2):
    show(axes[0,j], axes[1,j], img, title)
ymax = max(np.mean(p[1],axis=0).max() for p in panels2)
for j in range(3): axes[1,j].set_ylim(0, ymax*1.05)
plt.tight_layout(); plt.show()

for title,img in panels2:
    print(f'{title:26s} seam steps: {seam_steps(img)}')


In [ ]:
# ==========================================================================================
# CELL 3 — blending strategies across the overlap (the microscopy-stitch standard)
#
# Nikon NIS-Elements drives the ND2/YX1 path, NOT Keyence (Keyence = BZ-X Analyzer), so there is
# no Elements-specific setting to match here. But the universal microscopy-stitch default — used by
# Elements, Fiji Grid/Collection stitching, and ASHLAR — is LINEAR (feather) blending across the
# overlap: within the overlap band, weight each tile by its distance from its own edge so one tile
# fades out as the next fades in. That removes the hard seam cut regardless of residual intensity
# offset. (Our pipeline currently uses stitch2d, which does per-tile GAMMA matching but NO spatial
# feather — hence the hard steps.)
#
# Compared here:
#   (a) Hard cut          — average in the overlap (what `composite` did): a discrete boundary.
#   (b) Linear feather    — distance-weighted linear cross-fade across the overlap. RAW tiles.
#   (c) Linear feather + tile-wide gain — feather applied AFTER the (2) tile-wide intensity match.
# ==========================================================================================

def composite_feather(corrected_tiles):
    """Linear (feather) blend across each overlap: weight = distance from the tile's own edge."""
    acc = np.zeros((H, W)); wsum = np.zeros((H, W))
    for i, t in enumerate(corrected_tiles):
        x0 = i * step
        # per-column weight: ramps 0->1 across the LEFT overlap, flat 1 in the middle, 1->0 across
        # the RIGHT overlap. Edge tiles keep full weight on their outer side.
        w = np.ones(tile_w)
        if i > 0:                     # left overlap fades IN
            w[:overlap] = np.linspace(0, 1, overlap)
        if i < len(corrected_tiles)-1:  # right overlap fades OUT
            w[-overlap:] = np.linspace(1, 0, overlap)
        w2d = np.broadcast_to(w, (H, tile_w))
        acc[:, x0:x0+tile_w]  += t * w2d
        wsum[:, x0:x0+tile_w] += w2d
    wsum[wsum == 0] = 1
    return acc / wsum

hardcut       = composite(tiles)                                   # (a)
feather_raw   = composite_feather(tiles)                           # (b)
feather_gain  = composite_feather([g2[i]*tiles[i] for i in range(len(tiles))])  # (c)

panels3 = [('(a) Hard cut (current-style)', hardcut),
           ('(b) Linear feather, raw', feather_raw),
           ('(c) Feather + tile-wide gain', feather_gain)]

fig, axes = plt.subplots(2, 3, figsize=(18, 8), gridspec_kw={'height_ratios':[3,1.4]})
for j,(title,img) in enumerate(panels3):
    show(axes[0,j], axes[1,j], img, title)
ymax = max(np.mean(p[1],axis=0).max() for p in panels3)
for j in range(3): axes[1,j].set_ylim(0, ymax*1.05)
plt.tight_layout(); plt.show()

for title,img in panels3:
    print(f'{title:32s} seam steps: {seam_steps(img)}')


## CELL 4 — the MATERIALIZED pipeline output vs the notebook's feathered version

Loads the FF/projection image the pipeline actually wrote for `A02` and overlays its intensity
profile on the notebook's `feather + tile-wide gain` composite.

**Finding:** the shipped image behaves like a HARD CUT, not like the feathered version. The
discontinuity does **not** sit at the overlap *centre* (where our earlier `seam_steps()` check
measured, and passed) — it sits at the overlap **exit**, where the left tile ends.


In [ ]:
# ==========================================================================================
# CELL 4 — materialized pipeline output vs notebook feather+gain
#
# Geometry note: the materializer TRANSPOSES the horizontal strip to portrait and center-crops it
# to the legacy canvas (1710x720). So to compare against the notebook frame (H=720, W=2304) we
# un-transpose and account for the crop offset.
# ==========================================================================================
MAT_FF = (f'{ROOT}/pipeline_output/acquisition/20260702_hotchem_24hpf_plate01/materialized_images/'
          f'{well}/BF/projection/focus_stack/{well}_BF_t0000.png')

mat = skio.imread(MAT_FF)
if mat.ndim == 3: mat = mat[..., 0]
mat_h = mat.astype(float).T                 # portrait -> horizontal frame (720, 1710)
CROP  = (W - mat_h.shape[1]) // 2           # trim_to_shape center-crop offset (297)
print(f'materialized FF {mat.shape} -> un-transposed {mat_h.shape}, crop_start={CROP}')

# Landmarks in the notebook's full-canvas x coords
overlap_centers = seam_positions()                      # [816, 1488]  <- old check measured HERE
overlap_exits   = [i*step + tile_w for i in range(len(tiles)-1)]   # [960, 1632] <- real seam step
print(f'overlap centers: {overlap_centers}   overlap exits: {overlap_exits}')

nb_feather = composite_feather([g2[i]*tiles[i] for i in range(len(tiles))])
nb_hard    = composite(tiles)

def norm(p):
    p = np.asarray(p, float)
    lo, hi = np.percentile(p, 1), np.percentile(p, 99)
    return (p - lo) / (hi - lo + 1e-9)

# profiles on the common full-canvas x axis
x_mat = np.arange(mat_h.shape[1]) + CROP
p_mat = norm(mat_h.mean(axis=0))
p_fea = norm(nb_feather.mean(axis=0))
p_hrd = norm(nb_hard.mean(axis=0))

fig, axes = plt.subplots(2, 1, figsize=(15, 9), gridspec_kw={'height_ratios': [2.2, 1]})
axes[0].imshow(mat_h, cmap='gray', extent=[CROP, CROP+mat_h.shape[1], mat_h.shape[0], 0])
axes[0].set_title('MATERIALIZED FF (un-transposed to notebook frame)')
axes[0].set_xlim(0, W); axes[0].axis('off')

ax = axes[1]
ax.plot(np.arange(W), p_hrd, lw=1.0, color='0.6', label='notebook HARD CUT')
ax.plot(np.arange(W), p_fea, lw=1.4, color='tab:green', label='notebook FEATHER + gain')
ax.plot(x_mat, p_mat, lw=1.4, color='tab:red', label='MATERIALIZED FF (pipeline output)')
for i, s in enumerate(overlap_centers):
    ax.axvline(s, color='b', ls=':', lw=1.2, label='overlap CENTER (old check)' if i == 0 else None)
    axes[0].axvline(s, color='b', ls=':', lw=1.0)
for i, e in enumerate(overlap_exits):
    ax.axvline(e, color='r', ls='--', lw=1.2, label='overlap EXIT (real seam)' if i == 0 else None)
    axes[0].axvline(e, color='r', ls='--', lw=1.0)
ax.set_xlim(0, W); ax.set_xlabel('position along stitch axis (full-canvas px)')
ax.set_ylabel('normalized mean intensity'); ax.legend(loc='upper right', fontsize=9)
ax.set_title('Intensity profile — materialized output tracks the HARD CUT, not the feathered version')
plt.tight_layout(); plt.show()

# ---- quantify at BOTH landmarks ----
def step_at(prof, pos, off=0, k=6):
    i = int(round(pos - off))
    if i-k < 0 or i+k >= len(prof): return float('nan')
    return float(np.median(prof[i+1:i+k]) - np.median(prof[i-k:i-1]))

print(f"\n{'':28s} {'overlap CENTERS':>22s}   {'overlap EXITS':>22s}")
for name, prof, off in [('notebook HARD CUT', p_hrd, 0),
                        ('notebook FEATHER+GAIN', p_fea, 0),
                        ('MATERIALIZED FF', p_mat, CROP)]:
    c = [round(abs(step_at(prof, s, off))*100, 2) for s in overlap_centers]
    e = [round(abs(step_at(prof, s, off))*100, 2) for s in overlap_exits]
    print(f'{name:28s} {str(c):>22s}   {str(e):>22s}   (% of dyn range)')
print('\n-> Feathering fades tiles IN but never fades them OUT, so the step lands at the EXIT.')
